# 規則だけでチャットボットを作る（ELIZA 方式）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/B_implementations/eliza_rule.ipynb)

NLP クックブック B-19 ／ 仕組みを理解するための実装

機械学習を使わずに、パターンと応答テンプレートだけで対話するボットを作りたい。対話システムの原点を体験したい。

In [1]:
import random
import re

RULES = [
    (r"(.+)(?:が|は)(?:つらい|辛い|しんどい|苦しい)", ["{0}がつらいのですね。いつ頃からですか？", "{0}について、もう少し聞かせてください。"]),
    (r"(.+)(?:が|を)(?:心配|不安)", ["{0}の何がいちばん心配ですか？", "{0}が不安なのですね。"]),
    (r"(?:私|わたし|僕|ぼく)は(.+)(?:と思う|と思います)", ["なぜ{0}と思うのですか？", "本当に{0}でしょうか？"]),
    (r"(.+)したい", ["{0}したいのですね。何がそれを妨げていますか？", "{0}できたら、どんな気持ちになりそうですか？"]),
    (r"(母|父|家族|上司|先生|友達|友人)", ["{0}との関係について教えてください。", "{0}はあなたにとってどんな存在ですか？"]),
    (r"(?:いつも|毎日|ずっと)", ["「いつも」ですか。例外はありませんでしたか？"]),
    (r"(?:はい|うん|そうです)", ["なるほど。続けてください。"]),
    (r"(?:いいえ|いや|違い)", ["そうではないのですね。では、どうなのでしょう？"]),
    (r"(?:ありがとう|さようなら|またね)", ["お話しできてよかったです。またいつでもどうぞ。"]),
]
DEFAULT = ["もう少し詳しく話してもらえますか？", "それについてどう感じていますか？", "なるほど。それで？"]

def respond(text, rng):
    for pattern, templates in RULES:           # 上から順に照合し、最初に合った規則を使う
        m = re.search(pattern, text)
        if m:
            groups = [re.sub(r"^(?:でも|けど|最近|実は|正直)[、,]?\s*", "", g).strip("、。 ") for g in m.groups()]
            return rng.choice(templates).format(*groups)
    return rng.choice(DEFAULT)

rng = random.Random(0)
for user in ["最近、仕事がつらいです。", "上司に毎日怒られています。", "私は自分が悪いと思う。", "転職したい。", "でも家族が心配で。", "今日は天気がいいですね。", "ありがとう。"]:
    print(f"あなた: {user}\nボット: {respond(user, rng)}")

あなた: 最近、仕事がつらいです。
ボット: 仕事について、もう少し聞かせてください。
あなた: 上司に毎日怒られています。
ボット: 上司はあなたにとってどんな存在ですか？
あなた: 私は自分が悪いと思う。
ボット: なぜ自分が悪いと思うのですか？
あなた: 転職したい。
ボット: 転職できたら、どんな気持ちになりそうですか？
あなた: でも家族が心配で。
ボット: 家族が不安なのですね。
あなた: 今日は天気がいいですね。
ボット: それについてどう感じていますか？
あなた: ありがとう。
ボット: お話しできてよかったです。またいつでもどうぞ。


- 1966 年の ELIZA と同じ仕組み。相手の言葉を拾って質問で返すだけだが、「聞いてもらえている」感じが出る（ELIZA 効果）。
- 規則に合う入力には確実に意図どおりの応答を返せる。予約受付や定型的な案内では、今でも規則ベースの部分が残っている。
- 規則の順序が重要。特殊な規則を上に、一般的な規則を下に置く。どれにも合わないときの応答を必ず用意する。